(paired-estimation-and-discovery-where-the-predictive-gap-comes-from)=
# Paired estimation and discovery gaps

A predictive gap has two possible sources. The method can estimate the right function
badly, or it can fail to find the right structure in the first place. This benchmark
separates the two on synthetic panels (each a simulated truth shape, such as linear, interaction or threshold) where the optimal predictor is known. Three arms
see the same samples:

- **A_supplied**: the estimator is given the correct specification. Its only job is to
  estimate the effects, the smoothing and the stopping point.
- **A_partial**: the estimator is given every candidate predictor as a smooth term, and
  nothing about interactions. This is the weaker starting specification.
- **B_discovery**: a frozen discovery procedure chooses the terms itself, with a fixed
  resource budget.

The protocol is `docs/benchmarks/paired_estimation_discovery_protocol.md`. The run
outputs are in `docs/benchmarks/paired_ab/`: `report.txt`, which holds the summary,
and `results.json`, which holds one record per replicate and arm.

## Decomposition

Let $R_A$ and $R_B$ be the test risks of two arms, and $R^*$ the risk of the optimal
predictor, which is known on synthetic panels. Each gap is divided by $s$, the variance
of the response on the panel, so the gaps are comparable across panels:

$$
\text{estimation excess} = \frac{R_A - R^*}{s}, \qquad
\text{discovery gap} = \frac{R_B - R_A}{s}, \qquad
\text{partial-knowledge gap} = \frac{R_P - R_A}{s}, \qquad
\text{end-to-end excess} = \frac{R_B - R^*}{s}.
$$

The estimation excess and the discovery gap sum to the end-to-end excess. The sum is an
accounting identity, so it does not show which mechanism causes the gap.

## Precision rule

Each gap has a paired confidence half-width, computed from the 8 replicate gaps with a
t distribution on 7 degrees of freedom. A gap is resolved only when its median is larger
in absolute value than its half-width. A gap that is smaller than its half-width is not
resolved at this replicate count, and the protocol says more replicates are needed
before any claim about it.

In [1]:
import json
from pathlib import Path

import polars as pl
from great_tables import GT

# This page runs from docs/book/benchmarks; the v1.1 summary is the first JSON object in report.txt.
REPORT = Path("evidence/paired_ab/report.txt")
text = REPORT.read_text()
header_line, body = text.split("\n", 1)
summary, _ = json.JSONDecoder().raw_decode(body)
print(header_line)
print("panels:", list(summary.keys()))

paired-estimation-discovery paired-ab/v1.1
panels: ['linear_factors', 'smooth_additive', 'kinked', 'interaction', 'threshold_jump']


The table reports each gap's median over the 8 replicates, its paired half-width, and
whether the gap is resolved under the precision rule. Bold marks the resolved gaps.

In [2]:
gap_names = ["estimation_excess", "discovery_gap", "partial_knowledge_gap", "end_to_end_excess"]
gap_rows = []
for panel, values in summary.items():
    for gap in gap_names:
        entry = values[gap]
        median, half = entry["median"], entry["paired_ci_halfwidth"]
        resolved = abs(median) > half
        text_value = f"{median:+.4f} ± {half:.4f}"
        gap_rows.append(
            {
                "Panel": panel,
                "Gap": gap.replace("_", " "),
                "Median ± half-width": f"**{text_value}**" if resolved else text_value,
                "Resolved": "yes" if resolved else "no",
            }
        )

gap_table = pl.DataFrame(gap_rows)
(
    GT(gap_table)
    .tab_header(
        title="Predictive gaps by panel, v1.1 run",
        subtitle="Gap in units of Var(y), median over 8 replicates ± paired half-width; positive means worse than the first arm",
    )
    .fmt_markdown(columns=["Median ± half-width"])
    .tab_stubhead(label="Panel")
    .tab_source_note(
        "Bold: the median exceeds the paired half-width, so the gap is resolved. Source: the first JSON object in "
        "docs/benchmarks/paired_ab/report.txt, the paired-ab/v1.1 summary."
    )
)

GT(_tbl_data=shape: (20, 4)
┌─────────────────┬───────────────────────┬──────────────────────┬──────────┐
│ Panel           ┆ Gap                   ┆ Median ± half-width  ┆ Resolved │
│ ---             ┆ ---                   ┆ ---                  ┆ ---      │
│ str             ┆ str                   ┆ str                  ┆ str      │
╞═════════════════╪═══════════════════════╪══════════════════════╪══════════╡
│ linear_factors  ┆ estimation excess     ┆ +0.0006 ± 0.0019     ┆ no       │
│ linear_factors  ┆ discovery gap         ┆ -0.0001 ± 0.0005     ┆ no       │
│ linear_factors  ┆ partial knowledge gap ┆ +0.0004 ± 0.0010     ┆ no       │
│ linear_factors  ┆ end to end excess     ┆ -0.0001 ± 0.0019     ┆ no       │
│ smooth_additive ┆ estimation excess     ┆ -0.0008 ± 0.0051     ┆ no       │
│ …               ┆ …                     ┆ …                    ┆ …        │
│ interaction     ┆ end to end excess     ┆ +0.0526 ± 0.0651     ┆ no       │
│ threshold_jump  ┆ estimation excess     ┆ +0.0058 ± 0.0104     ┆ no       │
│ threshold_jump  ┆ discovery gap         ┆ +0.0060 ± 0.0061     ┆ no       │
│ threshold_jump  ┆ partial knowledge gap ┆ **+0.0089 ± 0.0067** ┆ yes      │
│ threshold_jump  ┆ end to end excess     ┆ **+0.0123 ± 0.0114** ┆ yes      │
└─────────────────┴───────────────────────┴──────────────────────┴──────────┘, _body=<great_tables._gt_data.Body object at 0x1093088f0>, _boxhead=Boxhead([ColInfo(var='Panel', type=<ColInfoTypeEnum.default: 1>, column_label='Panel', column_align='left', column_width=None), ColInfo(var='Gap', type=<ColInfoTypeEnum.default: 1>, column_label='Gap', column_align='left', column_width=None), ColInfo(var='Median ± half-width', type=<ColInfoTypeEnum.default: 1>, column_label='Median ± half-width', column_align='left', column_width=None), ColInfo(var='Resolved', type=<ColInfoTypeEnum.default: 1>, column_label='Resolved', column_align='left', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x10ac6eb40>, _spanners=Spanners([]), _heading=Heading(title='Predictive gaps by panel, v1.1 run', subtitle='Gap in units of Var(y), median over 8 replicates ± paired half-width; positive means worse than the first arm', preheader=None), _stubhead='Panel', _summary_rows=<great_tables._gt_data.SummaryRows object at 0x10a6059a0>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x107ab3c80>, _source_notes=['Bold: the median exceeds the paired half-width, so the gap is resolved. Source: the first JSON object in docs/benchmarks/paired_ab/report.txt, the paired-ab/v1.1 summary.'], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x10a14bf80>, _formats=[<great_tables._gt_data.FormatInfo object at 0x10ac3d220>], _substitutions=[], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_

On the linear, smooth and kinked panels, no gap is resolved: every interval covers zero.
On the interaction panel, two gaps are resolved. The partial-knowledge gap is 0.075 with
a half-width of 0.033, so the all-smooth starting specification loses about 7.5% of the
variance relative to the supplied specification. The discovery gap is −0.015 with a
half-width of 0.009, so the discovery arm does slightly better than the supplied
specification in the median. On the threshold panel, the partial-knowledge gap and the
end-to-end excess are resolved, at about 0.9% and 1.2% of the variance.

## Structure found by the discovery procedure

The discovery arm is scored on how many of the true structures it recovers (recall) and
how many of its selections are false (the false discovery proportion, FDP). The table
gives the median over 8 replicates.

In [3]:
structure_rows = []
for panel, values in summary.items():
    structure_rows.append(
        {
            "Panel": panel,
            "Growth steps, A_supplied": values["growth_steps_A_supplied"]["median"],
            "Boundary rate, A_supplied": values["boundary_rate_A_supplied"]["median"],
            "Growth steps, A_partial": values["growth_steps_A_partial"]["median"],
            "Boundary rate, A_partial": values["boundary_rate_A_partial"]["median"],
        }
    )

stopping = pl.DataFrame(structure_rows)
(
    GT(stopping)
    .tab_header(
        title="Stopping behaviour of the supplied-specification arms",
        subtitle="Median over 8 replicates; growth steps: budget doublings; boundary rate: share of replicates whose minimum sits at the cap",
    )
    .fmt_number(columns=[c for c in stopping.columns if c != "Panel"], decimals=2)
    .tab_stubhead(label="Panel")
    .tab_source_note("Source: the first JSON object in docs/benchmarks/paired_ab/report.txt.")
)

GT(_tbl_data=shape: (5, 5)
┌─────────────────┬───────────────┬────────────────┬───────────────┬──────────────────────────┐
│ Panel           ┆ Growth steps, ┆ Boundary rate, ┆ Growth steps, ┆ Boundary rate, A_partial │
│ ---             ┆ A_supplied    ┆ A_supplied     ┆ A_partial     ┆ ---                      │
│ str             ┆ ---           ┆ ---            ┆ ---           ┆ f64                      │
│                 ┆ f64           ┆ f64            ┆ f64           ┆                          │
╞═════════════════╪═══════════════╪════════════════╪═══════════════╪══════════════════════════╡
│ linear_factors  ┆ 0.0           ┆ 0.0            ┆ 0.0           ┆ 0.0                      │
│ smooth_additive ┆ 1.5           ┆ 0.0            ┆ 1.5           ┆ 0.0                      │
│ kinked          ┆ 0.0           ┆ 0.0            ┆ 0.0           ┆ 0.0                      │
│ interaction     ┆ 0.0           ┆ 0.0            ┆ 0.0           ┆ 0.0                      │
│ threshold_jump  ┆ 0.0           ┆ 0.0            ┆ 0.0           ┆ 0.0                      │
└─────────────────┴───────────────┴────────────────┴───────────────┴──────────────────────────┘, _body=<great_tables._gt_data.Body object at 0x10a24ef30>, _boxhead=Boxhead([ColInfo(var='Panel', type=<ColInfoTypeEnum.default: 1>, column_label='Panel', column_align='left', column_width=None), ColInfo(var='Growth steps, A_supplied', type=<ColInfoTypeEnum.default: 1>, column_label='Growth steps, A_supplied', column_align='right', column_width=None), ColInfo(var='Boundary rate, A_supplied', type=<ColInfoTypeEnum.default: 1>, column_label='Boundary rate, A_supplied', column_align='right', column_width=None), ColInfo(var='Growth steps, A_partial', type=<ColInfoTypeEnum.default: 1>, column_label='Growth steps, A_partial', column_align='right', column_width=None), ColInfo(var='Boundary rate, A_partial', type=<ColInfoTypeEnum.default: 1>, column_label='Boundary rate, A_partial', column_align='right', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x10acd2f90>, _spanners=Spanners([]), _heading=Heading(title='Stopping behaviour of the supplied-specification arms', subtitle='Median over 8 replicates; growth steps: budget doublings; boundary rate: share of replicates whose minimum sits at the cap', preheader=None), _stubhead='Panel', _summary_rows=<great_tables._gt_data.SummaryRows object at 0x109b1bb00>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x10993f1a0>, _source_notes=['Source: the first JSON object in docs/benchmarks/paired_ab/report.txt.'], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x109b1bce0>, _formats=[<great_tables._gt_data.FormatInfo object at 0x109500980>], _substitutions=[], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), tabl

Recall and FDP for the discovery arm are in the structure section of `report.txt`. On the interaction panel, recall is 1.00.



## Findings

- **On linear, smooth and kinked truth, no gap is resolved.** With eight replicates, the
  half-widths are as large as the gaps themselves, so this run cannot distinguish those
  gaps from zero.
- **On the interaction panel, the starting specification costs about 7.5% of the
  variance.** The partial-knowledge gap is resolved. The discovery gap is resolved in the
  negative direction, so discovery does not lose ground there in the median.
- **On the threshold panel, the partial-knowledge gap and the end-to-end excess are
  resolved.** The discovery gap is not: its median is 0.6% with a half-width of 0.6%.
- **The interval width, not the median, limits what this run can say.** Resolving a gap
  of 1% of the variance would need more replicates on the interaction panel, and the
  protocol states that a narrower gap calls for more replicates rather than weaker claims.

## Limits

- **Five synthetic panels.** The run covers linear, smooth additive, kinked, interaction
  and threshold truth. It has no spatial, location-scale or real-data panels.
- **Eight replicates per panel.** The half-widths are the precision of this run only.
- **One frozen discovery configuration.** The discovery gap is conditional on the frozen
  budget and on the adoption rule. It does not measure discovery in general.
- **Calibration is not measured.** The protocol reserves a calibration partition for
  interval calibration, which this run does not use.
- **Negative estimates near zero are noise.** Some estimation excesses are negative on
  the kinked panel. They are test noise at 2000 test rows, not negative excess.

## Reproduction

The protocol's provenance section gives the instrument version, the seeds and the
partition digests. The per-replicate records in `results.json` contain the test risks
and the terms selected by each arm.